In [ ]:
library(rsample)     # data splitting 
library(dplyr)       # data wrangling
library(rpart)       # performing regression trees
library(rpart.plot)  # plotting regression trees
library(ipred)       # bagging
library(caret)       # bagging
library(tidyverse)

In [ ]:
set.seed(123)
merged <- read_csv("data/merged_data.csv")
split <- initial_split(merged, prop = 0.75)
train <- training(split)
test  <- testing(split)


In [ ]:
library(rpart)

# 1. Drop non-numeric / ID columns
train_model <- train[, !names(train) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(train_model) <- make.names(names(train_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = train_model, method = "anova")



In [ ]:
fit

In [ ]:
set.seed(123)
merged <- read_csv("data/merged_data.csv")
ames_split <- initial_split(merged, prop = 0.75)
ames_train <- training(ames_split)
ames_test  <- testing(ames_split)

In [ ]:
library(rpart)

# 1. Drop non-numeric / ID columns
merged_model <- merged[, !names(merged) %in% c("County")]

# 2. Sanitize column names — this is the key fix
names(merged_model) <- make.names(names(merged_model), unique = TRUE)

# 3. Fit the tree
fit <- rpart(outbreak ~ ., data = merged_model, method = "anova", control = rpart.control(minsplit = 10, cp = 0.005))
fit

In [ ]:
hyper_grid <- expand.grid(
  minsplit = seq(5, 20, 1),
  maxdepth = seq(8, 15, 1)
)

head(hyper_grid)

nrow(hyper_grid)

In [ ]:

models <- list()

for (i in 1:nrow(hyper_grid)) {
  
  # get minsplit, maxdepth values at row i
  minsplit <- hyper_grid$minsplit[i]
  maxdepth <- hyper_grid$maxdepth[i]

  # train a model and store in the list
  models[[i]] <- rpart(
    formula = outbreak ~ .,
    data    = train_model,
    method  = "anova",
    control = list(minsplit = minsplit, maxdepth = maxdepth)
    )
}

In [ ]:
get_cp <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  cp <- x$cptable[min, "CP"] 
}

# function to get minimum error
get_min_error <- function(x) {
  min    <- which.min(x$cptable[, "xerror"])
  xerror <- x$cptable[min, "xerror"] 
}

hyper_grid %>%
  mutate(
    cp    = purrr::map_dbl(models, get_cp),
    error = purrr::map_dbl(models, get_min_error)
    ) %>%
  arrange(error) %>%
  top_n(-5, wt = error)

In [ ]:
# Make data input into predict
test_model <- test[, !names(test) %in% c("County")]
names(test_model) <- make.names(names(test_model), unique = TRUE)

optimal_tree <- rpart(
    formula = outbreak ~ .,
    data    = merged_model,
    method  = "anova",
    control = list(minsplit = 5, maxdepth = 9, cp = 0.0100000)
    )
optimal_tree
pred <- predict(optimal_tree, newdata = test_model)
RMSE(pred = pred, obs = test_model$outbreak)
#pred

In [ ]:
# WIP -----------------------------

# Load required libraries
library(tree)
library(ISLR)

# Set seed for reproducibility
set.seed(2)

# Remove the 'County' column FIRST, before sampling
merged_clean <- merged[, !names(merged) %in% c("County")]

tree=tree(outbreak ~ . -County , merged ,subset=merged)


# other WIP-------------------------------

#install.packages("smotefamily")
#library(smotefamily)

#train <- train[, !names(train) %in% c("County")]
#train <- na.omit(train)
#train <- train[, colSums(is.na(train)) == 0]

#names(train)[!sapply(train, is.numeric)]

#smote_output <- SMOTE(X = train[, names(train) != "outbreak"], 
                      #target = train$outbreak)